# Data coverage map

A one-page visual answer to the question *"what does SuSSE have data on,
and where?"*. Reads the warehouse live, so re-running this notebook
gives the current snapshot of what's been ingested.

Three things end up on the map:

| Marker | What it is | Source |
|---|---|---|
| **Blue dots** (sized by coverage) | Ground-measurement stations — the training labels | `ground_measurements` table |
| **Red dots** | Katongole 2023 validation stations | `notebooks/papers/mukiibi_mikelson_2026/reference_data/katongole_2023_monthly.csv` |
| **Green rectangle** | Uganda 2024 inference grid (portal-coverage cache) | `irradiance_daily` filtered to 2024-only cells outside ground-station geohashes |

Click any marker for the station's name + per-source date range and
day counts. The layer-control toggle in the top right lets you turn
individual layers on / off.

This notebook is **read-only** w.r.t. the warehouse — it only issues
SELECTs, no writes. Safe to re-run as often as you like.

## 0 — Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import folium
import pandas as pd
from folium import CircleMarker, FeatureGroup, LayerControl, Rectangle

# Resolve project root and put src/ on the path so `import susse` works
# regardless of where Jupyter was launched.
_NB_DIR = Path.cwd().resolve()
_PROJECT_ROOT = _NB_DIR
while _PROJECT_ROOT != _PROJECT_ROOT.parent and not (_PROJECT_ROOT / "src" / "susse").exists():
    _PROJECT_ROOT = _PROJECT_ROOT.parent
sys.path.insert(0, str(_PROJECT_ROOT / "src"))

from susse.warehouse_ops.io import BigQueryClient, WarehouseConfig
from susse.warehouse_ops.io.config import TableRefs

bq = BigQueryClient(config=WarehouseConfig())
tables = TableRefs(config=bq.config)
print(f"project root: {_PROJECT_ROOT}")
print(f"warehouse: {bq.config.project_id}.{bq.config.dataset}")

project root: /home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation
warehouse: solar-irradiation-estimation.solar_warehouse


## 1 — Load each data layer

### 1.1 Ground-measurement stations (training labels)

In [2]:
# One row per training station with its lat/lon, geohash5, and
# per-source satellite-coverage counts (number of distinct dates that
# have NASA or CAMS rows at the station's geohash5).
ground = bq.query(f"""
WITH ground_summary AS (
  SELECT location,
         ANY_VALUE(lat) AS lat,
         ANY_VALUE(lon) AS lon,
         ANY_VALUE(geohash5) AS geohash5,
         MIN(date) AS min_date,
         MAX(date) AS max_date,
         COUNT(*) AS n_rows,
         COUNTIF(qc_level = 'pass') AS n_qc_passed
  FROM `{tables.ground_measurements}`
  GROUP BY location
),
sat_coverage AS (
  SELECT geohash5,
         COUNT(DISTINCT IF(source='NASA', date, NULL)) AS n_nasa_days,
         COUNT(DISTINCT IF(source='CAMS', date, NULL)) AS n_cams_days
  FROM `{tables.irradiance_daily}`
  GROUP BY geohash5
)
SELECT g.*, COALESCE(s.n_nasa_days, 0) AS n_nasa_days,
            COALESCE(s.n_cams_days, 0) AS n_cams_days
FROM ground_summary g
LEFT JOIN sat_coverage s USING (geohash5)
ORDER BY n_rows DESC
""")
print(f"Ground stations: {len(ground)}")
ground.head()

/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Ground stations: 28


,location,lat,lon,geohash5,min_date,max_date,n_rows,n_qc_passed,n_nasa_days,n_cams_days
0,kenya_location3,-0.469974,35.181874,kzcj2,2019-05-28,2024-11-25,1976,1976,2009,2009
1,lira,2.295190,32.921370,s8rmj,2014-08-27,2023-02-01,1935,1934,3081,3081
2,kampala,0.333542,32.568630,s8p1v,2011-04-06,2023-01-22,1798,1798,4676,4676
3,ghana_location1,5.645759,-0.105223,ecpbj,2019-09-24,2024-11-25,1774,1774,1890,1890
4,nigeria_location1,9.076439,7.425385,s1t78,2020-04-04,2024-11-25,1641,1641,1697,1697


### 1.2 Katongole 2023 validation stations

In [8]:
import pygeohash

KATONGOLE_CSV = (
    _PROJECT_ROOT
    / "notebooks" / "papers" / "mukiibi_mikelson_2026"
    / "reference_data" / "katongole_2023_monthly.csv"
)
katongole = pd.read_csv(KATONGOLE_CSV)
katongole["geohash5"] = [
    pygeohash.encode(lat, lon, precision=5)
    for lat, lon in zip(katongole["latitude"], katongole["longitude"])
]

# Per-station 2017-2022 coverage (the climatology window the
# recomputation validates against).
_gh_quoted = ", ".join(f"'{g}'" for g in katongole["geohash5"].unique())
cov_2017_22 = bq.query(f"""
SELECT geohash5,
       COUNT(DISTINCT IF(source='NASA', date, NULL)) AS n_nasa_days,
       COUNT(DISTINCT IF(source='CAMS', date, NULL)) AS n_cams_days
FROM `{tables.irradiance_daily}`
WHERE date BETWEEN DATE('2017-01-01') AND DATE('2022-12-31')
  AND geohash5 IN ({_gh_quoted})
GROUP BY geohash5
""")
katongole = katongole.merge(cov_2017_22, on="geohash5", how="left").fillna(
    {"n_nasa_days": 0, "n_cams_days": 0}
).astype({"n_nasa_days": int, "n_cams_days": int})

# Bucketise so the map can colour-code coverage state.
_N_EXPECTED = (pd.Timestamp("2022-12-31") - pd.Timestamp("2017-01-01")).days + 1
_FULL = int(0.95 * _N_EXPECTED)
katongole["coverage_status"] = [
    "full" if (n_nasa >= _FULL and n_cams >= _FULL)
    else "missing" if (n_nasa == 0 and n_cams == 0)
    else "partial"
    for n_nasa, n_cams in zip(katongole["n_nasa_days"], katongole["n_cams_days"])
]
print(f"Katongole stations: {len(katongole)}")
print("Coverage breakdown (2017-2022 NASA + CAMS):")
print(katongole["coverage_status"].value_counts().to_string())

/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Katongole stations: 56
Coverage breakdown (2017-2022 NASA + CAMS):
coverage_status
full       41
partial    15


### 1.3 Uganda 2024 inference grid

In [9]:
# Pull the bbox + cell count of the 2024-only Uganda inference grid.
# Excludes ground-station cells (those have multi-year coverage from
# the A6 ingest, not just 2024).
uganda_grid = bq.query(f"""
SELECT MIN(latitude)  AS min_lat,
       MAX(latitude)  AS max_lat,
       MIN(longitude) AS min_lon,
       MAX(longitude) AS max_lon,
       COUNT(DISTINCT geohash5) AS n_cells,
       MIN(date) AS min_date,
       MAX(date) AS max_date
FROM `{tables.irradiance_daily}`
WHERE date BETWEEN DATE('2024-01-01') AND DATE('2024-12-31')
  AND source = 'NASA'
  AND geohash5 NOT IN (
    SELECT DISTINCT geohash5 FROM `{tables.ground_measurements}`
  )
""")
print(uganda_grid.to_string(index=False))

/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 min_lat  max_lat   min_lon   max_lon  n_cells   min_date   max_date
-1.38214  4.21786 29.671499 34.971499     1961 2024-01-01 2024-12-31


## 2 — Build the map

Each layer is a `FeatureGroup`; the `LayerControl` widget in the top-
right of the map lets the reader toggle them independently. Markers
have HTML popups with the station's name + coverage summary.

Map tiles default to OpenStreetMap (no API key, no per-tile cost).
Switch to a satellite basemap with the `tiles=` argument if you'd
rather see the terrain underneath.

In [10]:
def _ground_popup(row: pd.Series) -> str:
    return (
        f"<b>{row.location}</b><br>"
        f"lat={row.lat:.3f}, lon={row.lon:.3f} (geohash5 {row.geohash5})<br>"
        f"<b>Ground</b>: {row.n_qc_passed:,} QC-passed days "
        f"({row.min_date} → {row.max_date})<br>"
        f"<b>NASA</b>:   {row.n_nasa_days:,} days<br>"
        f"<b>CAMS</b>:   {row.n_cams_days:,} days"
    )


def _katongole_popup(row: pd.Series) -> str:
    if row.coverage_status == "full":
        cov = "<span style='color:green'>full 2017-2022 coverage</span>"
    elif row.coverage_status == "missing":
        cov = "<span style='color:red'>no warehouse data yet</span> — run migration A12"
    else:
        cov = (
            f"<span style='color:orange'>partial</span>: "
            f"NASA {row.n_nasa_days} days, CAMS {row.n_cams_days} days"
        )
    return (
        f"<b>{row.location}</b><br>"
        f"lat={row.latitude:.3f}, lon={row.longitude:.3f}<br>"
        f"{cov}"
    )


# Centre roughly on Sub-Saharan Africa; the 28 ground stations span
# Egypt to Madagascar, so a continent-wide initial zoom is the right
# default — readers can zoom in to the Uganda detail after.
m = folium.Map(location=[5.0, 25.0], zoom_start=4, control_scale=True)

# Layer 1 — Ground stations.
ground_layer = FeatureGroup(name=f"Ground stations ({len(ground)})", show=True)
for _, row in ground.iterrows():
    # Marker radius scales with the log of QC-passed days so very-long
    # stations don't dominate the visual.
    radius = 4 + int(0.001 * row.n_qc_passed)
    CircleMarker(
        location=[row.lat, row.lon],
        radius=min(radius, 14),
        popup=folium.Popup(_ground_popup(row), max_width=320),
        tooltip=row.location,
        color="#1f77b4",
        weight=1,
        fill=True,
        fillColor="#1f77b4",
        fillOpacity=0.8,
    ).add_to(ground_layer)
ground_layer.add_to(m)

# Layer 2 — Katongole validation stations, colour by warehouse coverage.
katongole_layer = FeatureGroup(
    name=f"Katongole validation ({len(katongole)})", show=True,
)
_status_color = {"full": "#2ca02c", "partial": "#ff7f0e", "missing": "#d62728"}
for _, row in katongole.iterrows():
    color = _status_color[row.coverage_status]
    CircleMarker(
        location=[row.latitude, row.longitude],
        radius=4,
        popup=folium.Popup(_katongole_popup(row), max_width=320),
        tooltip=f"{row.location} — {row.coverage_status}",
        color=color,
        weight=1,
        fill=True,
        fillColor=color,
        fillOpacity=0.85,
    ).add_to(katongole_layer)
katongole_layer.add_to(m)

# Layer 3 — Uganda 2024 inference-grid bbox.
ug_layer = FeatureGroup(name="Uganda 2024 inference grid", show=True)
ug_row = uganda_grid.iloc[0]
Rectangle(
    bounds=[
        [float(ug_row.min_lat), float(ug_row.min_lon)],
        [float(ug_row.max_lat), float(ug_row.max_lon)],
    ],
    color="#2ca02c",
    weight=2,
    fill=True,
    fillColor="#2ca02c",
    fillOpacity=0.08,
    popup=folium.Popup(
        f"<b>Uganda 2024 inference grid</b><br>"
        f"{int(ug_row.n_cells):,} cells × 365 days<br>"
        f"{ug_row.min_date} → {ug_row.max_date}",
        max_width=320,
    ),
    tooltip="Uganda 2024 inference grid",
).add_to(ug_layer)
ug_layer.add_to(m)

LayerControl(collapsed=False).add_to(m)
m

## 3 — Quick reference tables

Same information the map carries, in tabular form — useful for
copy-pasting into reports or filtering further.

In [6]:
# Ground stations, sorted by ground-truth coverage.
ground_view = ground[[
    "location", "lat", "lon", "geohash5",
    "min_date", "max_date", "n_qc_passed",
    "n_nasa_days", "n_cams_days",
]].copy()
ground_view.style.set_caption("Ground stations (training labels)")

,location,lat,lon,geohash5,min_date,max_date,n_qc_passed,n_nasa_days,n_cams_days
0,kenya_location3,-0.469974,35.181874,kzcj2,2019-05-28,2024-11-25,1976,2009,2009
1,lira,2.295190,32.921370,s8rmj,2014-08-27,2023-02-01,1934,3081,3081
2,kampala,0.333542,32.568630,s8p1v,2011-04-06,2023-01-22,1798,4676,4676
3,ghana_location1,5.645759,-0.105223,ecpbj,2019-09-24,2024-11-25,1774,1890,1890
4,nigeria_location1,9.076439,7.425385,s1t78,2020-04-04,2024-11-25,1641,1697,1697
5,kenya_location5,-0.220000,35.860000,kzcw8,2020-12-22,2024-11-25,1382,1435,1435
6,tororo,0.697800,34.171510,sb07c,2011-03-08,2017-11-22,1380,4317,4317
7,ghana_location3,-0.235111,5.632167,kpun8,2021-02-25,2024-11-25,1320,1370,1370
8,ghana_location2,5.645686,0.008678,s1000,2020-01-31,2024-05-02,1105,1554,1554
9,kenya_location2,0.610000,36.800000,sb45m,2019-10-08,2024-02-23,1016,1600,1600


In [7]:
# Katongole stations grouped by warehouse coverage status.
katongole_view = katongole.groupby("coverage_status").agg(
    n_stations=("location", "size"),
    median_nasa_days=("n_nasa_days", "median"),
    median_cams_days=("n_cams_days", "median"),
)
print("Katongole 2017-2022 warehouse coverage summary:")
print(katongole_view.to_string())
print()
incomplete = katongole[katongole["coverage_status"] != "full"].sort_values(
    "coverage_status"
)
if not incomplete.empty:
    print(f"Stations not yet at full coverage ({len(incomplete)}):")
    print(
        incomplete[["location", "coverage_status", "n_nasa_days", "n_cams_days"]]
        .to_string(index=False)
    )

Katongole 2017-2022 warehouse coverage summary:
                 n_stations  median_nasa_days  median_cams_days
coverage_status                                                
full                     39            2191.0            2191.0
partial                  17            2191.0               0.0

Stations not yet at full coverage (17):
   location coverage_status  n_nasa_days  n_cams_days
   Yumbe HQ         partial         2191            0
  Sembabule         partial         2191            0
 Mateete SC         partial         2191            0
   Rakai HQ         partial         2191            0
Kibaale CAI         partial         2191            0
  Kibanda H         partial         2191            0
 Perfect IP         partial         2191            0
Lwebitakuli         partial         2191            0
   Kyambogo         partial         2191            0
   Mengo SS         partial         2191            0
  Kisasi PS         partial         2191            0
     Ai

## What's not on the map (yet)

* **MERRA-2 coverage** — `merra_daily_vars_long` is populated for the
  Uganda 2024 bbox (post-`b9` migration), but the cells aren't visualised
  separately because they share footprint with the green rectangle.
  Add a fourth layer if you need per-source MERRA visibility.
* **MODIS observations** — composite-cadence, not daily, so the
  visualisation would need a date slider. Out of scope for this overview.
* **Per-day animation.** A `folium.plugins.TimeSliderChoropleth` over
  `irradiance_daily` would show how coverage evolved over time. Worth
  building as a follow-up if the static map turns out to be too coarse.